# POLG rare variant extraction from CES data

- **Project**: Multi-ancestry analysis of POLG variants in Parkinson’s disease
- **Last Update:** APRIL-2026

## Set Paths

In [ ]:
import pathlib
## GP2 v11.0 Path
REL11_PATH = pathlib.Path(pathlib.Path.home(), "/path/to/gp2/release11")
!ls -hal {REL11_PATH}

In [ ]:
## Define file paths for clinical and genetic data
EXTENDED_CLINICAL_DATA_PATH = pathlib.Path(REL11_PATH, 'clinical_data/extended_clinical_data_release11_vwb.csv')
CLINICAL_DATA_PATH          = pathlib.Path(REL11_PATH, 'clinical_data/master_key_release11_final_vwb.csv')
#RELATED_DATA_PATH          = pathlib.Path(REL11_PATH, 'meta_data/related_samples/')
#RAW_GENO_PATH              = pathlib.Path(REL11_PATH, 'raw_genotypes')
#IMPUTED_GENO_PATH          = pathlib.Path(REL11_PATH, 'imputed_genotypes')
#PCS_PATH                   = pathlib.Path(REL11_PATH, 'imputed_genotypes')

## Install packages

### Install Slivar

In [ ]:
%%bash 

#install Slivar

#mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/slivar; then
    echo "Slivar is already installed in /home/jupyter/tools/"
else
    echo -e "Downloading Slivar \n    -------"
    wget https://github.com/brentp/slivar/releases/download/v0.2.8/slivar
    chmod +x ./slivar
    wget https://raw.githubusercontent.com/brentp/slivar/master/js/slivar-functions.js
    wget https://slivar.s3.amazonaws.com/gnomad.hg38.genomes.v3.fix.zip
    echo -e "\n Slivar downloaded and unzipped in /home/jupyter/tools \n "

fi

In [ ]:
# check if the Slivar is installed successfully 

! ~/tools/slivar


### Install bcftools

In [ ]:
%%bash 

#install bcftools

#mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/bcftools; then
    echo "bcftools is already installed in /home/jupyter/tools/"
else
    echo -e "Downloading bcftools \n    -------"
    git clone --recurse-submodules https://github.com/samtools/htslib.git
    git clone https://github.com/samtools/bcftools.git
    cd bcftools
    make
    echo -e "\n bcftools downloaded and unzipped in /home/jupyter/tools \n "

fi


In [ ]:
# check bcftools 
!/home/jupyter/tools/bcftools/bcftools --help

In [ ]:
%%bash
## Define the working directory
# Create a folder on your workspace
#WORK_DIR = !mkdir f'/home/jupyter/workspace/POLG_results/vcfs'
#mkdir /home/jupyter/workspace/POLG_results/vcfs
cd /home/jupyter/workspace/POLG_results/vcfs

In [ ]:
# check what's in the working directory
! ls /home/jupyter/workspace/POLG_results/vcfs

### Install PLINK

In [ ]:
%%capture
%%bash

# Install plink 1.9
cd /home/jupyter/
if test -e /home/jupyter/plink; then
    echo "Plink is already installed in /home/jupyter/"
else
    echo "Plink is not installed"
    cd /home/jupyter

    wget http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 

    unzip -o plink_linux_x86_64_20190304.zip
    mv plink plink1.9
fi

In [ ]:
%%bash

# chmod plink 1.9 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink1.9

In [ ]:
%%capture
%%bash

# Install plink 2.0
cd /home/jupyter/
if test -e /home/jupyter/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/

wget http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

unzip -o plink2_linux_x86_64_latest.zip

fi

In [ ]:
%%bash

# chmod plink 2 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink2

## Define the gene you want to work with

In [ ]:
gene     = 'POLG'      # your gene name
chrom    = '15'        # number of chromosome
bp_start = '89305198'  # transcript start
bp_end   = '89334861'  # transcript end

## Define PF exome vcf pathes

In [ ]:
#exome-vcfs= pathlib.Path(REL8_PATH,workspace/gp2_tier2_eu_release8_13092024/clinical_exomes/deepvariant_joint_calling/vcfs/chr{chrom}.vcf.gz)
exome_vcfs = pathlib.Path(REL8_PATH, "clinical_exomes/deepvariant_joint_calling/vcfs/chr{chrom}.vcf.gz")

In [ ]:
#download ped file
exome_pgen= pathlib.Path(REL8_PATH,"clinical_exomes/deepvariant_joint_calling/plink/all_chrs.pgen")
#shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp {WORKSPACE_BUCKET}/exome-vcfs/PDGENE_final.ped {WORK_DIR}')

#download slivar js file
#shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp {WORKSPACE_BUCKET}/exome-vcfs/slivar-functions.v0.2.8.js ~/tools/')
silvar_path = '~/tools/slivar-functions.v0.2.8.js'

In [ ]:
!/home/jupyter/plink2 --pgen {REL8_PATH}/clinical_exomes/deepvariant_joint_calling/plink/chr15.pgen \
       --psam {REL8_PATH}/clinical_exomes/deepvariant_joint_calling/plink/chr15.psam \
       --pvar {REL8_PATH}/clinical_exomes/deepvariant_joint_calling/plink/chr15.pvar \
       --recode ped \
       --out /home/jupyter/workspace/POLG_results/vcfs/chr15

In [ ]:
exome_ped= pathlib.Path("/home/jupyter/workspace/POLG_results/vcfs/chr15.ped")

In [ ]:
!ls /home/jupyter/workspace/POLG_results/vcfs
#!cat {WORK_DIR}/chr15.ped | head -n 20

# Extract rare variants in the gene

## Extract variants in the gene

In [ ]:
# Format the input and output file paths correctly
input_vcf = exome_vcfs.with_name(f"chr{chrom}.vcf.gz")
output_vcf = pathlib.Path.cwd() / f"{gene}.vcf.gz"  # Output in the current working directory


In [ ]:
# use bcftools to extract the variants in the gene
!{"/home/jupyter/tools/bcftools/bcftools"} view -r chr{chrom}:{bp_start}-{bp_end} \
    -Oz -o {output_vcf} \
    {input_vcf}

In [ ]:
#!zcat {output_vcf} | head -n 20
!zcat {WORK_DIR}/{gene}.vcf.gz | head -n 20

# Filter for variants with gnomAD_Popmax_AF < 0.005 

In [ ]:
!/home/jupyter/tools/slivar expr --vcf {WORK_DIR}/{gene}.vcf.gz  --ped {exome_ped} --js /home/jupyter/tools/slivar-functions.js

In [ ]:
WORK_DIR = "/home/jupyter/workspace/POLG_results/vcfs"  # Update as necessary
output_file = f"{WORK_DIR}/{gene}_filtered.vcf"
#gnomad= pathlib.Path("/home/jupyter/tools/gnomad.hg38.genomes.v3.fix.zip")
#!file /home/jupyter/tools/gnomad.hg38.genomes.v3.fix.zip

In [ ]:
!ls -l /home/jupyter/tools/slivar-functions.js
#!chmod 644 /home/jupyter/tools/slivar-functions.js
#!chmod 644 /home/jupyter/tools/gnomad.hg38.genomes.v3.fix.zip

In [ ]:
!/home/jupyter/tools/slivar expr \
    --vcf /home/jupyter/workspace/POLG_results/vcfs/POLG.vcf.gz \
    --ped /home/jupyter/workspace/POLG_results/vcfs/chr15.ped \
    --js /home/jupyter/tools/slivar-functions.js \
    -g /home/jupyter/tools/gnomad.hg38.genomes.v3.fix.zip \
    --pass-only \
    --info "INFO.AF < 0.005 && variant.ALT[0] != '*' && variant.call_rate >= 0.95" \
    --family-expr "HOM:fam.every(function(s) {return s.hom_alt && s.affected && hq1(s)})" \
    --family-expr "HET:fam.every(function(s) {return s.het && s.affected && hq1(s)})" \
    --family-expr "comphet_side:fam.every(function(s) {(s.het || s.hom_ref) && hq1(s)}) && fam.some(function(s) {return s.het && s.affected}) && INFO.gnomADg_nhomalt <= 1" \
> /home/jupyter/workspace/POLG_results/vcfs/POLG_filtered.vcf

In [ ]:
# check for potential comphet variants
!/home/jupyter/tools/slivar compound-hets \
                    --vcf {WORK_DIR}/{gene}_filtered.vcf \
                    --ped {exome_ped}\
                    --allow-non-trios --sample-field comphet_side --sample-field denovo > {WORK_DIR}/{gene}_comphet.vcf
            

In [ ]:
# check for potential comphet variants
!/home/jupyter/tools/slivar compound-hets \
                    --vcf {WORK_DIR}/{gene}_filtered.vcf \
                    --ped {WORK_DIR}/chr15.ped \
                    --allow-non-trios --sample-field comphet_side --sample-field denovo > {WORK_DIR}/{gene}_comphet.vcf

## Format the results to tsv

In [ ]:
!cat /home/jupyter/workspace/POLG_results/vcfs/POLG_filtered.vcf | grep -m 1 "##INFO=<ID=CSQ"

In [ ]:
info_fields = [
    'AF',
    'AC',
    'AN',
    # Add other available fields as needed
]

csq_columns = [
    'Allele',
    'Consequence',
    'IMPACT',
    'SYMBOL',
    'Gene',
    'Feature_type',
    'Feature',
    'BIOTYPE',
    'EXON',
    'INTRON',
    'HGVSc',
    'HGVSp',
    'cDNA_position',
    'CDS_position',
    'Protein_position',
    'Amino_acids',
    'Codons',
    'Existing_variation',
    'DISTANCE',
    'STRAND',
    'FLAGS',
    'VARIANT_CLASS',
    'SYMBOL_SOURCE',
    'HGNC_ID',
    'CANONICAL',
    'MANE_SELECT',
    'MANE_PLUS_CLINICAL',
    'TSL',
    'APPRIS',
    'CCDS',
    'ENSP',
    'SWISSPROT',
    'TREMBL',
    'UNIPARC',
    'UNIPROT_ISOFORM',
    'GENE_PHENO',
    'NEAREST',
    'SIFT',
    'PolyPhen',
    'DOMAINS',
    'miRNA',
    'HGVS_OFFSET',
    'gnomADe_AF',
    'gnomADg_AF',
    'MAX_AF',
    'MAX_AF_POPS',
    'CLIN_SIG',
    'SOMATIC',
    'PHENO',
    'PUBMED',
    'VAR_SYNONYMS',
    'MOTIF_NAME',
    'MOTIF_POS',
    'HIGH_INF_POS',
    'MOTIF_SCORE_CHANGE',
    'TRANSCRIPTION_FACTORS'
]



In [ ]:
# Build command list

command = [
    '/home/jupyter/tools/slivar', 'tsv',
    '-s', 'slivar_comphet',
]

# Add info fields
for field in info_fields:
    command.append(f"--info-field")
    command.append(field)

# Add CSQ columns
for column in csq_columns:
    command.append(f"--csq-column")
    command.append(column)

# Add other options
command.extend([
    '-c', 'CSQ',
    '-p', f'{WORK_DIR}/chr15.ped',
    f'{WORK_DIR}/{gene}_comphet.vcf'
])

# Execute the command and filter output
output_file = f'{WORK_DIR}/comphet.tsv'
with open(output_file, 'w') as outfile:
    process = subprocess.run(command, stdout=subprocess.PIPE, check=True, text=True)
    
    # Filter out header lines
    for line in process.stdout.splitlines():
        if not line.startswith('#'):
            outfile.write(line + '\n')
    


In [ ]:
! head {WORK_DIR}/POLG_filtered.tsv

In [ ]:
# make filtered tsv (HET and HOM variants)
# make filtered tsv (HET and HOM vaimport subprocess

info_fields = [
    'AF',
    'AC',
    'AN',
    # Add other available fields as needed
]

csq_columns = [
    'Allele',
    'Consequence',
    'IMPACT',
    'SYMBOL',
    'Gene',
    'Feature_type',
    'Feature',
    'BIOTYPE',
    'EXON',
    'INTRON',
    'HGVSc',
    'HGVSp',
    'cDNA_position',
    'CDS_position',
    'Protein_position',
    'Amino_acids',
    'Codons',
    'Existing_variation',
    'DISTANCE',
    'STRAND',
    'FLAGS',
    'VARIANT_CLASS',
    'SYMBOL_SOURCE',
    'HGNC_ID',
    'CANONICAL',
    'MANE_SELECT',
    'MANE_PLUS_CLINICAL',
    'TSL',
    'APPRIS',
    'CCDS',
    'ENSP',
    'SWISSPROT',
    'TREMBL',
    'UNIPARC',
    'UNIPROT_ISOFORM',
    'GENE_PHENO',
    'NEAREST',
    'SIFT',
    'PolyPhen',
    'DOMAINS',
    'miRNA',
    'HGVS_OFFSET',
    'gnomADe_AF',
    'gnomADg_AF',
    'MAX_AF',
    'MAX_AF_POPS',
    'CLIN_SIG',
    'SOMATIC',
    'PHENO',
    'PUBMED',
    'VAR_SYNONYMS',
    'MOTIF_NAME',
    'MOTIF_POS',
    'HIGH_INF_POS',
    'MOTIF_SCORE_CHANGE',
    'TRANSCRIPTION_FACTORS'
]

# Build command list
command = [
    '/home/jupyter/tools/slivar', 'tsv',
]

# Add info fields
for field in info_fields:
    command.append(f"--info-field")
    command.append(field)

# Add CSQ columns
for column in csq_columns:
    command.append(f"--csq-column")
    command.append(column)

# Add other options
command.extend([
    '-s', 'HOM',
    '-s', 'HET',
    '-c', 'CSQ',
    '-p', f'{WORK_DIR}/chr15.ped',
    f'{WORK_DIR}/{gene}_filtered.vcf'
])

# Execute the command and redirect output to a file
with open(f'{WORK_DIR}/{gene}_filtered.tsv', 'w') as outfile:
    subprocess.run(command, stdout=outfile, check=True)


In [ ]:
# combine comphet and filtered 
!cat {WORK_DIR}/{gene}_filtered.tsv {WORK_DIR}/{gene}_comphet.tsv| sed -r -e 's/slivar_comphet_[0-9]+/comphet/g' > {WORK_DIR}/{gene}_all_vars.tsv

# have a look
!head {WORK_DIR}/{gene}_all_vars.tsv

## Processing the tsv

In [ ]:
df=pd.read_csv(f'{WORK_DIR}/{gene}_all_vars.tsv',sep='\t')
df=df.loc[df['gene']==gene]

df

In [ ]:
# function to find the annotation column

def get_max_str(lst):
    return max(lst, key=len)


In [ ]:
# split the annotation

all_csq_cols = ['gene','impact','transcript'] + csq_columns
column_to_move = get_max_str(df.columns)
df[column_to_move] = df.pop(column_to_move)

anno= df[get_max_str(df.columns)].str.split('/', expand=True).add_prefix('ann')
anno.rename(columns=dict(zip(list(anno.columns)[0:len(all_csq_cols)],all_csq_cols)),inplace=True)
tmp=pd.concat([df,anno],axis=1)
tmp.drop(get_max_str(df.columns), axis=1, inplace=True)
tmp=tmp.loc[:,~tmp.columns.duplicated()]

tmp

In [ ]:
# check coding variants

coding = tmp.loc[tmp['IMPACT'].isin(['HIGH','MODERATE'])]
coding

In [ ]:
!pip install openpyxl

In [ ]:
# save the dataframe for review and pathogenicity scoring

#coding = coding.apply(lambda x: x.str.encode('utf-8').str.decode('utf-8') if x.dtype == "object" else x)

# Save to Excel
#coding.to_excel(f'{WORK_DIR}/{gene}_coding_vars.xlsx', index=False, engine='openpyxl')
coding.to_csv(f'{WORK_DIR}/{gene}_coding_vars.csv', index=False, encoding='utf-8')
